In [57]:
import sqlite3 
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool 
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config={
    "configurable":{
        "thread_id":"11",
    }
}

In [58]:

class State(MessagesState):
    pass

graph_builder = StateGraph(State)

In [59]:
def chatbot(state:State):
    response = llm.invoke(state["messages"])
    return {
        "messages" : [response]
    }

In [60]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot",END)


# 그래프 컴파일
# 그래프랑 edge들이 다~말이되는지화긘 (유효성검사)
graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn),
)



In [63]:
result = graph.invoke(
    {
        'messages': [{
            "role":"user", "content":"I live oin korea now."
        }]
    },
    config=config
)

In [64]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello my name is wish.
================================== Ai Message ==================================

Hello, Wish! How can I assist you today?
================================ Human Message =================================

I live oin korea now.
================================== Ai Message ==================================

That sounds great! Korea has a rich culture and beautiful landscapes. What do you enjoy most about living there?


In [68]:
state_history = graph.get_state_history(config)

In [69]:
for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print(state_snapshot.values["messages"])
    print("=========\n")

()
[HumanMessage(content='Hello my name is wish.', additional_kwargs={}, response_metadata={}, id='6b7de173-5c9a-45e1-8779-59e3de256191'), AIMessage(content='Hello, Wish! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 13, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DM91Am0uyUknI6KXjbOGrTCVETq9o', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d14c9-a790-7681-88e9-e7198c0caa3b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 11, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output

In [70]:
state_history = graph.get_state_history(config)

to_fork = list(state_history)[-5]

to_fork.values["messages"]

[HumanMessage(content='Hello my name is wish.', additional_kwargs={}, response_metadata={}, id='6b7de173-5c9a-45e1-8779-59e3de256191'),
 AIMessage(content='Hello, Wish! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 13, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DM91Am0uyUknI6KXjbOGrTCVETq9o', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d14c9-a790-7681-88e9-e7198c0caa3b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 11, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_t

In [71]:
from langchain_core.messages import HumanMessage

graph.update_state(
    to_fork.config, 
    # 이 시점으로 거슬러 올라가..
    {
        "messages": [HumanMessage(
            content = "I live in korea now. And the city I live in is seoul",
            id="e1a31818-8e8b-4414-abc3-3fc86289b885"
        )]
    }
)

{'configurable': {'thread_id': '11',
  'checkpoint_ns': '',
  'checkpoint_id': '1f125d01-d5f9-66d0-8004-58d0bfe75ed7'}}

In [72]:
forked_state = graph.get_state_history(
    {'configurable': {'thread_id': '11',
  'checkpoint_ns': '',
  'checkpoint_id': '1f125d01-d5f9-66d0-8004-58d0bfe75ed7'}}
)

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content='Hello my name is wish.', additional_kwargs={}, response_metadata={}, id='6b7de173-5c9a-45e1-8779-59e3de256191'), AIMessage(content='Hello, Wish! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 13, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DM91Am0uyUknI6KXjbOGrTCVETq9o', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d14c9-a790-7681-88e9-e7198c0caa3b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 11, 'total_tokens': 24, 'input_token_details': {'audi

In [73]:
result = graph.invoke(None,{'configurable': {'thread_id': '11',
  'checkpoint_ns': '',
  'checkpoint_id': '1f125d01-d5f9-66d0-8004-58d0bfe75ed7'}})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello my name is wish.
================================== Ai Message ==================================

Hello, Wish! How can I assist you today?
================================ Human Message =================================

I live in korea now. And the city I live in is seoul
================================== Ai Message ==================================

That's great! Seoul is a vibrant city with a rich culture and history. What do you enjoy most about living there?
